In [ ]:
# [목적] TextLoader로 뉴스 텍스트 파일을 불러와 요약에 사용할 문서 객체로 준비합니다.
# 환경 변수와 LangSmith 추적을 설정한 뒤 파일 내용을 docs에 저장합니다.
# 문서의 글자 수와 앞부분을 확인해 이후 요약 체인에 전달할 입력을 점검합니다.
from langchain_community.document_loaders import TextLoader
from langchain_teddynote import logging
from dotenv import load_dotenv

# .env 파일에 저장된 OpenAI·LangSmith API 키 등의 환경 변수를 불러옵니다.
load_dotenv()
# 이후 LangChain 실행 과정을 LangSmith에서 'Summary' 프로젝트로 추적합니다.
logging.langsmith("Summary")

# TextLoader는 일반 텍스트 파일을 LangChain의 Document 객체로 변환합니다.
loader = TextLoader(
    r"D:\shproject\ex0929\data\news.txt",
    encoding="utf-8"
)

# load() 결과는 문서 목록이며, 각 문서의 본문은 page_content에 들어 있습니다.
docs = loader.load()

print(f"총 글자수: {len(docs[0].page_content)}")
print("\n======= 앞부분 미리보기 =======\n")
print(docs[0].page_content[:500])

In [ ]:
# [목적] LangSmith Hub에서 한국어 문서 요약용 프롬프트를 불러와 구조를 확인합니다.
# Client로 공개 프롬프트를 가져온 뒤 사람이 읽기 쉬운 형태로 출력합니다.
# 불러온 prompt는 다음 셀에서 언어 모델과 결합해 요약 체인을 만드는 데 사용합니다.
from langsmith import Client

# Client는 LangSmith에 저장된 프롬프트를 조회하는 연결 객체입니다.
client = Client()

prompt = client.pull_prompt(
    "teddynote/summary-stuff-documents-korean",
    # LangSmith Hub에 공개된 프롬프트를 내려받을 수 있도록 허용합니다.
    dangerously_pull_public_prompt=True
)

prompt.pretty_print()

In [ ]:
# [목적] ChatOpenAI와 Stuff 문서 체인을 연결해 뉴스 문서를 스트리밍 방식으로 요약합니다.
# docs 전체를 prompt의 context에 넣고 모델을 실행하며, 생성되는 답변을 실시간으로 표시합니다.
# 완성된 요약 결과는 answer에 저장해 이후 확인하거나 다른 작업에 활용할 수 있습니다.
from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_teddynote.callbacks import StreamingCallback

# ChatOpenAI는 OpenAI 채팅 모델을 LangChain 체인에서 사용할 수 있게 감싼 객체입니다.
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    # streaming은 답변이 완성되기 전에도 생성된 내용을 순서대로 전달합니다.
    streaming=True,
    # 0에 가까울수록 결과의 무작위성을 줄여 일관된 요약을 만듭니다.
    temperature=0,
    # StreamingCallback은 전달받은 답변 조각을 Notebook 출력에 바로 표시합니다.
    callbacks=[StreamingCallback()],
)

# Stuff 체인은 모든 문서를 하나의 context로 묶어 prompt와 모델에 한 번에 전달합니다.
stuff_chain = create_stuff_documents_chain(llm, prompt)
# invoke의 context 키가 프롬프트의 문서 입력 위치와 연결됩니다.
answer = stuff_chain.invoke({"context": docs})

In [17]:
# [목적] 문서 요약 지시와 context 자리표시자를 포함한 사용자 정의 프롬프트를 만듭니다.
# PromptTemplate이 실행 시 전달받은 문서 내용을 {context} 위치에 넣도록 구성합니다.
# 이 prompt는 원하는 요약 규칙을 직접 작성해 요약 체인에 적용할 때 사용합니다.
from langchain_core.prompts import PromptTemplate

# from_template은 문자열 안의 중괄호 항목을 입력 변수로 인식합니다.
prompt = PromptTemplate.from_template(
    """ Please summarize the sentence according to the following REQUEST.
REQUEST:
...
CONTEXT:
{context}

SUMMARY:
"""
)